# 02_3 DQA Control Sweep 6h

02_2では、client pseudo-label trainingそのものがwarmupを壊しやすく、DQA/FedAvgの差はかなり小さいことが見えました。

このNotebookでは、追加のclient学習を増やさず、02_2で作ったclient checkpointsとpseudo statsを再利用して、DQAをどう制御すれば機能するのかを検証します。

主な検証:

- DQA hyperparameter sweep: server floor, residual blend, classwise blend, temperature, count/quality gate
- client subset ablation: overcast/rainy/snowy単独、leave-one-client-out、全client
- FedAvgとの差分: 同じclient subsetでDQAがFedAvgを超える条件
- warmup rollback rule: warmup未満なら採用しない判断表
- top候補だけserver calibrationを1 epoch実行

想定実行時間は、02_2のclient checkpointsが存在している状態で6時間以内です。出力は `dynamic_quality_aware_classwise_aggregation/exploring/runs/02_3_dqa_control_sweep_6h/` に保存します。

## 1. Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "Object_Detection" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT.name != "Object_Detection":
    raise RuntimeError("Run this notebook from inside /app/Object_Detection")

EXPLORING_ROOT = ROOT / "dynamic_quality_aware_classwise_aggregation" / "exploring"
if str(EXPLORING_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPLORING_ROOT))

from dqa_probe_02_3 import ControlSweepConfig, DQAControlSweep

cfg = ControlSweepConfig(
    source_experiment_name="02_2_dqa_functionality_probe_7h",
    experiment_name="02_3_dqa_control_sweep_6h",
    source_warmup_key="dqa_v2_scene_12h",
    max_wall_hours=6.0,
    server_train_limit=2048,
    server_val_limit=1024,
    client_target_limit=2048,
    device_cli="0",
    batch_size=8,
    val_batch_size=1,
    workers=0,
    force_rerun=False,
    run_top_calibration=True,
    top_calibration_k=8,
)

sweep = DQAControlSweep(cfg)
sweep.describe()

{'experiment_root': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/exploring/runs/02_3_dqa_control_sweep_6h',
 'source_experiment': '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/exploring/runs/02_2_dqa_functionality_probe_7h',
 'source_variants': ['ssod_default_all_lr1e-2',
  'ssod_default_all_lr1e-3',
  'ssod_strict_all_lr3e-4',
  'ssod_strict_neck_head_lr1e-3',
  'ssod_very_strict_all_lr3e-4'],
 'subsets': ['all_clients',
  'overcast_only',
  'rainy_only',
  'snowy_only',
  'drop_overcast',
  'drop_rainy',
  'drop_snowy'],
 'candidates': ['fedavg',
  'dqa_current',
  'dqa_floor60_resid20',
  'dqa_floor70_resid12',
  'dqa_floor80_resid05',
  'dqa_floor90_resid02',
  'dqa_count_gate100',
  'dqa_quality_gate30'],
 'max_wall_hours': 6.0}

## 2. Source Check

02_2のclient checkpointsとpseudo statsが揃っているか確認します。ここがmissingなら、先に02_2のclient trainingまで回してください。

In [2]:
source_status = sweep.source_status()
source_status

,variant,client_id,weather,checkpoint_exists,checkpoint,stats_exists,stats_path
0,ssod_default_all_lr1e-2,0,overcast,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
1,ssod_default_all_lr1e-2,1,rainy,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
2,ssod_default_all_lr1e-2,2,snowy,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
3,ssod_default_all_lr1e-3,0,overcast,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
4,ssod_default_all_lr1e-3,1,rainy,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
5,ssod_default_all_lr1e-3,2,snowy,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
6,ssod_strict_all_lr3e-4,0,overcast,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
7,ssod_strict_all_lr3e-4,1,rainy,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
8,ssod_strict_all_lr3e-4,2,snowy,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...
9,ssod_strict_neck_head_lr1e-3,0,overcast,True,/app/Object_Detection/dynamic_quality_aware_cl...,True,/app/Object_Detection/dynamic_quality_aware_cl...


In [3]:
pseudo = sweep.pseudo_summary()
cols = ["variant", "client_id", "weather", "pseudo_total", "active_classes", "zero_classes", "top_class_share", "mean_quality_active"]
cols = [c for c in cols if c in pseudo.columns]
pseudo[cols].sort_values(["variant", "client_id"])

,variant,client_id,weather,pseudo_total,active_classes,zero_classes,top_class_share,mean_quality_active
0,ssod_default_all_lr1e-2,0,overcast,115450.0,9,1,0.490515,0.508380
1,ssod_default_all_lr1e-2,1,rainy,93950.0,8,2,0.954582,0.481109
2,ssod_default_all_lr1e-2,2,snowy,122965.0,8,2,0.958720,0.503663
3,ssod_default_all_lr1e-3,0,overcast,118204.0,9,1,0.721194,0.510406
4,ssod_default_all_lr1e-3,1,rainy,313870.0,9,1,0.929493,0.459459
5,ssod_default_all_lr1e-3,2,snowy,135106.0,8,2,0.781290,0.508477
6,ssod_strict_all_lr3e-4,0,overcast,57058.0,8,2,0.755109,0.696148
7,ssod_strict_all_lr3e-4,1,rainy,42662.0,8,2,0.773428,0.687957
8,ssod_strict_all_lr3e-4,2,snowy,47458.0,8,2,0.749673,0.689709
9,ssod_strict_neck_head_lr1e-3,0,overcast,50471.0,8,2,0.791583,0.698297


## 3. Run Control Sweep

全client / 単独client / leave-one-outに対して、FedAvgと複数のDQA制御設定を評価します。途中で止まっても既存JSON/checkpointはskipします。

In [4]:
leaderboard = sweep.run_all()
display_cols = [
    "name", "kind", "variant", "subset", "candidate", "status",
    "map50_final", "map50_95_final", "delta_map50_95_vs_warmup", "delta_vs_fedavg",
    "precision_final", "recall_final", "source_table",
]
display_cols = [c for c in display_cols if c in leaderboard.columns]
leaderboard[display_cols].head(40)

,name,kind,variant,subset,candidate,...,delta_map50_95_vs_warmup,delta_vs_fedavg,precision_final,recall_final,source_table
280,cal_ssod_very_strict_all_lr3e-4__overcast_only...,control_top_calibration,ssod_very_strict_all_lr3e-4,overcast_only,dqa_count_gate100,...,NaN,NaN,0.61992,0.39297,top_calibration_summary.csv
287,cal_ssod_very_strict_all_lr3e-4__overcast_only...,control_top_calibration,ssod_very_strict_all_lr3e-4,overcast_only,dqa_floor70_resid12,...,NaN,NaN,0.62191,0.39238,top_calibration_summary.csv
286,cal_ssod_very_strict_all_lr3e-4__overcast_only...,control_top_calibration,ssod_very_strict_all_lr3e-4,overcast_only,dqa_floor90_resid02,...,NaN,NaN,0.62214,0.39231,top_calibration_summary.csv
281,cal_ssod_default_all_lr1e-2__snowy_only__dqa_f...,control_top_calibration,ssod_default_all_lr1e-2,snowy_only,dqa_floor60_resid20,...,NaN,NaN,0.62404,0.39103,top_calibration_summary.csv
282,cal_ssod_very_strict_all_lr3e-4__drop_snowy__d...,control_top_calibration,ssod_very_strict_all_lr3e-4,drop_snowy,dqa_quality_gate30,...,NaN,NaN,0.61988,0.39288,top_calibration_summary.csv
284,cal_ssod_default_all_lr1e-2__all_clients__dqa_...,control_top_calibration,ssod_default_all_lr1e-2,all_clients,dqa_current,...,NaN,NaN,0.62431,0.39083,top_calibration_summary.csv
285,cal_ssod_default_all_lr1e-2__all_clients__dqa_...,control_top_calibration,ssod_default_all_lr1e-2,all_clients,dqa_floor60_resid20,...,NaN,NaN,0.62041,0.39209,top_calibration_summary.csv
283,cal_ssod_very_strict_all_lr3e-4__drop_snowy__d...,control_top_calibration,ssod_very_strict_all_lr3e-4,drop_snowy,dqa_count_gate100,...,NaN,NaN,0.61969,0.39305,top_calibration_summary.csv
128,ssod_very_strict_all_lr3e-4__all_clients__dqa_...,control_aggregate_eval,ssod_very_strict_all_lr3e-4,all_clients,dqa_count_gate100,...,0.0,0.000,0.61700,0.38900,decision_table.csv
147,ssod_very_strict_all_lr3e-4__all_clients__fedavg,control_aggregate_eval,ssod_very_strict_all_lr3e-4,all_clients,fedavg,...,0.0,0.000,0.61600,0.38900,decision_table.csv


## 4. Decision Table

`decision_table.csv` は、DQAを採用すべきかの判断表です。

- `delta_map50_95_vs_warmup > 0`: warmupより良いので採用候補
- `delta_vs_fedavg > 0`: 同じclient subsetでFedAvgよりDQAが良い
- `subset`: どのweather clientが効いた/壊したかを見るための軸
- `candidate`: DQA制御設定

In [5]:
decision = sweep.build_decision_table()
decision_cols = [
    "variant", "subset", "candidate", "status", "map50_final", "map50_95_final",
    "delta_map50_95_vs_warmup", "delta_vs_fedavg", "accept_over_warmup", "accept_over_fedavg",
    "precision_final", "recall_final", "active_classes", "candidate_note",
]
decision_cols = [c for c in decision_cols if c in decision.columns]
decision[decision_cols].head(60)

,variant,subset,candidate,status,map50_final,...,accept_over_fedavg,precision_final,recall_final,active_classes,candidate_note
238,ssod_very_strict_all_lr3e-4,overcast_only,dqa_count_gate100,ok,0.439,...,True,0.616,0.388,6.0,Ignore classes with too little pseudo-label su...
26,ssod_default_all_lr1e-2,snowy_only,dqa_floor60_resid20,ok,0.436,...,True,0.565,0.428,8.0,Moderate server floor and smaller residuals.
279,ssod_very_strict_all_lr3e-4,drop_snowy,dqa_quality_gate30,ok,0.439,...,True,0.616,0.388,7.0,Ignore low-quality pseudo-label classes.
278,ssod_very_strict_all_lr3e-4,drop_snowy,dqa_count_gate100,ok,0.439,...,True,0.616,0.388,6.0,Ignore classes with too little pseudo-label su...
1,ssod_default_all_lr1e-2,all_clients,dqa_current,ok,0.437,...,True,0.610,0.394,8.0,Current v2-style DQA anchor.
2,ssod_default_all_lr1e-2,all_clients,dqa_floor60_resid20,ok,0.438,...,True,0.626,0.385,8.0,Moderate server floor and smaller residuals.
237,ssod_very_strict_all_lr3e-4,overcast_only,dqa_floor90_resid02,ok,0.439,...,True,0.616,0.388,7.0,Near-rollback DQA. Useful as a lower-risk acce...
235,ssod_very_strict_all_lr3e-4,overcast_only,dqa_floor70_resid12,ok,0.439,...,True,0.616,0.388,7.0,"Conservative DQA, close to a guarded client re..."
275,ssod_very_strict_all_lr3e-4,drop_snowy,dqa_floor70_resid12,ok,0.439,...,True,0.616,0.388,7.0,"Conservative DQA, close to a guarded client re..."
50,ssod_default_all_lr1e-2,drop_snowy,dqa_floor60_resid20,ok,0.437,...,True,0.611,0.392,8.0,Moderate server floor and smaller residuals.


## 5. Subset And Candidate Views

DQAが効く条件を見るため、variant/subset/candidateごとのbestをざっくり集計します。

In [6]:
import pandas as pd

if not decision.empty:
    best_by_subset = (
        decision.sort_values("map50_95_final", ascending=False)
        .groupby(["variant", "subset"], as_index=False)
        .first()
    )
    subset_cols = ["variant", "subset", "candidate", "map50_final", "map50_95_final", "delta_map50_95_vs_warmup", "delta_vs_fedavg"]
    display(best_by_subset[[c for c in subset_cols if c in best_by_subset.columns]].sort_values("map50_95_final", ascending=False).head(40))
else:
    print("decision table is empty")

,variant,subset,candidate,map50_final,map50_95_final,delta_map50_95_vs_warmup,delta_vs_fedavg
0,ssod_default_all_lr1e-2,all_clients,dqa_current,0.437,0.242,0.000,0.003
1,ssod_default_all_lr1e-2,drop_overcast,dqa_floor60_resid20,0.438,0.242,0.000,0.006
2,ssod_default_all_lr1e-2,drop_rainy,dqa_floor70_resid12,0.438,0.242,0.000,0.005
3,ssod_default_all_lr1e-2,drop_snowy,dqa_quality_gate30,0.438,0.242,0.000,0.005
4,ssod_default_all_lr1e-2,overcast_only,dqa_floor60_resid20,0.438,0.242,0.000,0.009
6,ssod_default_all_lr1e-2,snowy_only,dqa_floor90_resid02,0.439,0.242,0.000,0.012
7,ssod_default_all_lr1e-3,all_clients,dqa_quality_gate30,0.438,0.242,0.000,0.001
9,ssod_default_all_lr1e-3,drop_rainy,dqa_current,0.438,0.242,0.000,0.001
8,ssod_default_all_lr1e-3,drop_overcast,dqa_floor70_resid12,0.439,0.242,0.000,0.000
10,ssod_default_all_lr1e-3,drop_snowy,dqa_floor60_resid20,0.439,0.242,0.000,0.000


In [7]:
if not decision.empty:
    candidate_view = (
        decision.groupby("candidate", as_index=False)
        .agg(
            mean_map50_95=("map50_95_final", "mean"),
            best_map50_95=("map50_95_final", "max"),
            mean_delta_vs_fedavg=("delta_vs_fedavg", "mean"),
            accepted_over_warmup=("accept_over_warmup", "sum"),
        )
        .sort_values("best_map50_95", ascending=False)
    )
    candidate_view
else:
    print("decision table is empty")

## 6. Top Calibration

`run_all()` でtop候補だけserver calibration済みです。必要ならこのセルで再表示します。DQAがaggregation直後に弱くても、calibration後にwarmupを超えるなら、DQA後のserver repairを必須にする価値があります。

In [8]:
cal_path = sweep.result_root / "top_calibration_summary.csv"
if cal_path.exists():
    cal = pd.read_csv(cal_path)
    cal_cols = ["source_control_name", "variant", "subset", "candidate", "status", "map50_final", "map50_95_final", "precision_final", "recall_final"]
    cal[[c for c in cal_cols if c in cal.columns]].sort_values("map50_95_final", ascending=False)
else:
    print("No top calibration summary yet")

## 7. Interpretation

見るべき結論:

- DQAがFedAvgより勝つcandidateがあるか
- 全clientより単独/leave-one-outが良いなら、特定weather clientが更新を壊している
- `floor80/floor90` が強いなら、client residualはかなり小さくするべき
- `count_gate` / `quality_gate` が強いなら、pseudo statsによるclass gatingが必要
- top calibration後だけ良いなら、DQA aggregation単体ではなくDQA + server repairを1セットにする
- warmup未満しか出ないなら、phase2 client updateはrollback対象にする